In [2]:
import dask.array as da
import numpy as np

from calmd.utils import find_chunks, nan_quantile

In [13]:
arr_shape = (1000, 1800, 45000)
chnks = find_chunks(arr_shape, (1, None, None), max_size=40, dtype='float32')

In [14]:
darr = da.empty(shape=arr_shape, chunks=chnks, dtype='float32')

In [15]:
darr

dask.array<empty_like, shape=(1000, 1800, 45000), dtype=float32, chunksize=(1, 632, 15811), chunktype=numpy.ndarray>

In [18]:
for i in np.arange(1000):
    rng = np.random.default_rng()
    darr[i, :, :] = rng.random((1800, 45000))

MemoryError: Unable to allocate 309. MiB for an array with shape (1800, 45000) and data type float32

In [ ]:
rng

array([[0.04378138, 0.89890366, 0.51087919, ..., 0.91630164, 0.86964462,
        0.95038019],
       [0.40929618, 0.43939294, 0.99200461, ..., 0.64396245, 0.89913906,
        0.03808786],
       [0.60121696, 0.22505917, 0.67281708, ..., 0.274925  , 0.09845552,
        0.63217168],
       ...,
       [0.19591113, 0.61426666, 0.40971619, ..., 0.51934284, 0.13035071,
        0.72537768],
       [0.23290346, 0.64100222, 0.01936153, ..., 0.51596971, 0.42828983,
        0.96152851],
       [0.9335572 , 0.20913215, 0.85362111, ..., 0.94418498, 0.59326571,
        0.13470875]], shape=(1800, 45000))

In [127]:
t=darr[:,[0],:].compute()

In [128]:
t.shape

(1000, 1, 10000)

In [129]:
nan_quantile(t, [0.05, 0.95])

array([[[0.0551199 , 0.04582859, 0.04206214, ..., 0.05062702,
         0.05351312, 0.04693672]],

       [[0.9508667 , 0.93682581, 0.96056489, ..., 0.95860887,
         0.94601663, 0.92764764]]], shape=(2, 1, 10000))

In [112]:
np.nanquantile(t, [0.05, 0.95], axis=0)

array([[[0.04782167, 0.05808848, 0.04411951, ..., 0.04541616,
         0.0526733 , 0.06118445]],

       [[0.96733049, 0.95778551, 0.95003042, ..., 0.93817747,
         0.94668796, 0.95240437]]], shape=(2, 1, 2000))

In [114]:
qs = [0.05, 0.95]

In [173]:
daskquan = da.map_blocks(nan_quantile, darr, qs, chunks=(len(qs), 1, darr.chunksize[2]), dtype=dtype)

In [175]:
daskquan

dask.array<nan_quantile, shape=(2, 14000, 119), dtype=float32, chunksize=(2, 1, 119), chunktype=numpy.ndarray>

In [176]:
fin = daskquan.compute()

In [177]:
fin[0, :, 0]

array([0.05300441, 0.04779215, 0.04729209, ..., 0.05097041, 0.05359421,
       0.05051778], shape=(14000,))